In [ ]:
import os
import subprocess
from pathlib import Path
import re

LABEL = "QP0"
DATA_DIR = Path("/home/swnh/pgc/datasets/nuscenes/v1.0-mini/ply/bin")
ENCODER_BIN = Path("/home/swnh/gpcc/build/sandbox/predgeom")
EXP_DIR = Path("/home/swnh/gpcc/experiments")

SCALE = 1000
GROUP = 1
RANGE = 1
BOUNDARY = 1
ANGULAR = 1
QP = 0

scene_averages = {}
total_size = 0
total_bpp = 0.0
total_frames = 0

print(f"Starting evaluation of mini set targeting: {ENCODER_BIN.name}")
for scene_dir in sorted(DATA_DIR.iterdir()):
    if not scene_dir.is_dir():
        continue

    scene_name = scene_dir.name
    scene_out_dir = EXP_DIR / LABEL / scene_name
    scene_out_dir.mkdir(parents=True, exist_ok=True)

    scene_sizes = []
    scene_bpps = []
    # Iterate over ply files
    for ply_file in sorted(scene_dir.glob("*.ply")):
        bitstream_path = scene_out_dir / f"{ply_file.stem}.bin"
        decoded_path   = scene_out_dir / f"{ply_file.stem}_decoded.ply"

        # Fixed Error 1: Convert integers to strings
        # Fixed Error 2: Use '--groups' instead of '--group'
        cmd = [str(ENCODER_BIN),
               "--input"   , str(ply_file),
               "--output"  , str(bitstream_path),
               "--decoded" , str(decoded_path),
               "--scale"   , str(SCALE),
               "--groups"  , str(GROUP),
               "--range"   , str(RANGE),
               "--boundary", str(BOUNDARY),
               "--flat-qp" , str(QP),
               "--angular" , str(ANGULAR)"]
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            match = re.search(r"Payload size:\s+(\d+)\s+B\s+\(([\d.]+)\s+bpp\)", result.stdout)
            if match:
                scene_sizes.append(int(match.group(1)))
                scene_bpps.append(float(match.group(2)))
        except subprocess.CalledProcessError:
            pass

    if scene_sizes:
        avg_size = sum(scene_sizes) / len(scene_sizes)
        avg_bpp = sum(scene_bpps) / len(scene_bpps)
        scene_averages[scene_name] = avg_size
        total_size += sum(scene_sizes)
        total_bpp += sum(scene_bpps)
        total_frames += len(scene_sizes)
        print(f"Scene: {scene_name} - Avg Payload: {avg_size:8.2f} B, Avg bpp: {avg_bpp:5.2f} ({len(scene_sizes):2d} frames)")

if total_frames > 0:
    overall_avg = total_size / total_frames
    overall_avg_bpp = total_bpp / total_frames
    print("-" * 50)
    print(f"Overall Average Payload Size across {len(scene_averages)} scenes ({total_frames} frames): {overall_avg:.2f} B")
    print(f"Overall Avg Payload: {overall_avg:.2f} B, Overall Avg bpp: {overall_avg_bpp:.2f}")
else:
    print("No frames processed.")

Starting evaluation of mini set targeting: predgeom
No frames processed.


In [5]:
import numpy as np
from plyfile import PlyData

verts = PlyData.read("/home/swnh/pgc/datasets/nuscenes/v1.0-mini/ply/bin/scene-0757/scene-0757_00.ply")["vertex"].data
x   = np.asarray(verts["x"])
y   = np.asarray(verts["y"])
ring = np.asarray(verts["ring"]).astype(np.int32)

phi = np.arctan2(y, x)

print(f"{'ring':>4} {'count':>6} {'step°':>8} {'N_rev':>7}")
for r in np.unique(ring):
    mask = ring == r
    p = np.sort(phi[mask])
    if len(p) < 2:
        continue
    dphi = np.diff(p)
    dphi = dphi[dphi > 0]           # drop duplicates / wrap
    if len(dphi) == 0:
        continue
    step = np.median(dphi)          # median is robust to the wrap gap
    n    = 2 * np.pi / step
    print(f"{r:4d} {mask.sum():6d} {np.rad2deg(step):8.3f} {n:7.0f}")


ring  count    step°   N_rev
   0   1083    0.308    1170
   1   1083    0.316    1138
   2   1083    0.321    1120
   3   1083    0.324    1111
   4   1083    0.324    1110
   5   1083    0.327    1100
   6   1083    0.329    1093
   7   1083    0.330    1090
   8   1083    0.333    1081
   9   1083    0.333    1082
  10   1083    0.332    1083
  11   1083    0.332    1083
  12   1083    0.332    1083
  13   1083    0.332    1083
  14   1083    0.332    1085
  15   1083    0.332    1083
  16   1083    0.332    1085
  17   1083    0.332    1085
  18   1083    0.331    1088
  19   1083    0.331    1089
  20   1083    0.331    1088
  21   1083    0.330    1090
  22   1083    0.330    1090
  23   1083    0.330    1091
  24   1083    0.330    1091
  25   1083    0.330    1091
  26   1083    0.330    1090
  27   1083    0.330    1092
  28   1083    0.325    1109
  29   1083    0.045    8022
  30   1083    0.002  225244
  31   1083    0.003  116351


In [1]:
import os
import subprocess
from pathlib import Path
import re

DATA_DIR = Path("/home/swnh/pgc/datasets/nuscenes/v1.0-mini/ply/bin")
ENCODER_BIN_MVP = Path("/home/swnh/gpcc/mvp/build/app")
OUT_BIN = Path("/tmp/test_out.bin")

scene_averages = {}
total_size = 0
total_frames = 0

print(f"Starting evaluation of MVP set targeting: {ENCODER_BIN_MVP.name}")
for scene_dir in sorted(DATA_DIR.iterdir()):
    if not scene_dir.is_dir():
        continue
        
    scene_name = scene_dir.name
    scene_sizes = []
    
    # Iterate over ply files
    for ply_file in sorted(scene_dir.glob("*.ply")):
        cmd = [str(ENCODER_BIN_MVP), "--input", str(ply_file), "--output", str(OUT_BIN)]
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            match = re.search(r"bitstream size\s+(\d+)\s+B", result.stdout)
            if match:
                scene_sizes.append(int(match.group(1)))
        except subprocess.CalledProcessError:
            pass
            
    if scene_sizes:
        avg_size = sum(scene_sizes) / len(scene_sizes)
        scene_averages[scene_name] = avg_size
        total_size += sum(scene_sizes)
        total_frames += len(scene_sizes)
        print(f"Scene: {scene_name} - Average Payload Size: {avg_size:8.2f} B ({len(scene_sizes):2d} frames)")

if total_frames > 0:
    overall_avg = total_size / total_frames
    print("-" * 50)
    print(f"Overall Average Payload Size across {len(scene_averages)} scenes ({total_frames} frames): {overall_avg:.2f} B")
else:
    print("No frames processed.")


Starting evaluation of MVP set targeting: app
Scene: scene-0061 - Average Payload Size: 95361.18 B (39 frames)
Scene: scene-0103 - Average Payload Size: 86603.35 B (40 frames)
Scene: scene-0553 - Average Payload Size: 81788.93 B (41 frames)
Scene: scene-0655 - Average Payload Size: 86408.71 B (41 frames)
Scene: scene-0757 - Average Payload Size: 86999.80 B (41 frames)
Scene: scene-0796 - Average Payload Size: 99216.62 B (40 frames)
Scene: scene-0916 - Average Payload Size: 100090.93 B (41 frames)
Scene: scene-1077 - Average Payload Size: 96983.39 B (41 frames)
Scene: scene-1094 - Average Payload Size: 100806.52 B (40 frames)
Scene: scene-1100 - Average Payload Size: 102173.15 B (40 frames)
--------------------------------------------------
Overall Average Payload Size across 10 scenes (404 frames): 93599.54 B
